In [ ]:
import pandas as pd
import random
import json
from tqdm import tqdm

In [ ]:
## OPEN FILE
file = 'path/to/raw/jsonl/file' # raw data (train.jsonl, valid.jsonl)
data = []
with open(file, 'r') as f:
    for line in f:
        data.append(json.loads(line.strip())) # read raw data into list

df = pd.DataFrame(data) # convert to pd.DataFrame

In [ ]:
dct = {'sentence' : [], 'cause' : [], 'effect': [], 'label' : [], 'input': []}

def generate_negative_example(example): # define neg example function. 'example' is a dictionary
    all_negative_events_within_two_sent = []
    for event in example['events']: # iterate over all events
        for event_mention in event['mention']: # iterate over the event 'mentions'
            for pot_event in example['events']: # iterate over potential candidate events to for negative effect
                for pot_event_mention in pot_event['mention']: # iterate over potential event mentions
                    # pass if events: are the same, form a positive example, or have the same trigger word
                    if event['id'] == pot_event['id']:
                        continue
                    if [event['id'], pot_event['id']] in example['causal_relations']['CAUSE'] + example['causal_relations']['PRECONDITION']:
                        continue
                    if event_mention['trigger_word'] == pot_event_mention['trigger_word']:
                        continue
                    else:
                        if 0 <= pot_event_mention['sent_id'] - event_mention['sent_id'] <= 2: # ensure events are within 2 sentences of each other
                            sentence = ' '.join(example['sentences'][event_mention['sent_id'] : pot_event_mention['sent_id']+1]) # form input string
                            all_negative_events_within_two_sent.append([event_mention, pot_event_mention, sentence]) # append to negative example candidates
                            if len(all_negative_events_within_two_sent) == 50: # limit potential candidates to 50
                                return random.choice(all_negative_events_within_two_sent) # randomly sample an event
    # if there are fewer than 50 candidates, check there are candidates, and sample randomly
    if len(all_negative_events_within_two_sent) > 0:
        return random.choice(all_negative_events_within_two_sent)
    return # if no candidates, return None


for _, example in tqdm(df.iterrows()):

    all_events = {event['id'] : event for event in example['events']} # get all events in the example

    for cause, effect in example['causal_relations']['CAUSE'] + example['causal_relations']['PRECONDITION']: # event1 -> cause, event2 -> effect, iterate over positive examples

        cause_all_mentions, effect_all_mentions = all_events[cause]['mention'], all_events[effect]['mention'] # gets the 'mentions' (may be mentioned multiple times)

        for cause_mention in cause_all_mentions: # iterate over the cause mentions
            for effect_mention in effect_all_mentions: # iterate over effect mentions
                cause_sent_id = cause_mention['sent_id'] # find sentence number of each event mention
                effect_sent_id = effect_mention['sent_id']

                if 0 <= abs(cause_sent_id - effect_sent_id) <= 2 and cause_mention['trigger_word'] != effect_mention['trigger_word']: # check both within 2 sentences, and not referred to by same word
                    # obtain input string for positive example
                    if cause_sent_id <= effect_sent_id:
                        sentence = ' '.join(example['sentences'][cause_sent_id : effect_sent_id+1])
                    else:
                        sentence = ' '.join(example['sentences'][effect_sent_id : cause_sent_id+1])
                    dct['cause'].append(cause_mention['trigger_word'])
                    dct['effect'].append(effect_mention['trigger_word'])
                    dct['sentence'].append(sentence)
                    dct['label'].append(1)
                    dct['input'].append(f"Given the input: '{dct['sentence'][-1]}', did '{dct['cause'][-1]}' cause '{dct['effect'][-1]}'?")


                    output = generate_negative_example(example) # obtain negative example
                    if output:
                        neg_cause, neg_effect, neg_sentence = output
                        dct['cause'].append(neg_cause['trigger_word'])
                        dct['effect'].append(neg_effect['trigger_word'])
                        dct['sentence'].append(neg_sentence)
                        dct['label'].append(0)
                        dct['input'].append(f"Given the input: '{dct['sentence'][-1]}', did '{dct['cause'][-1]}' cause '{dct['effect'][-1]}'?")

df2 = pd.DataFrame(dct) # convert dct to pd.Dataframe
df2.to_csv('T_standard.csv') # output to csv

In [ ]:
# check that dataset is balanced
pos = len([i for i in df2['label'] if i == 1])
neg = len([i for i in df2['label'] if i == 0])

print(pos)
print(neg)

In [ ]:
T_paraphrase_unique_examples = pd.read_csv("T_paraphrase_unique_examples.csv") # load unique examples

# training paraphrase formats P_train
P_train = [
    "Considering the input: '{input}' can we conclude that '{cause}' is responsible for causing '{effect}'?",
    "Given the input: '{input}', can '{cause}' be attributed to causing '{effect}'?",
    "Is '{cause}' a necessary condition for '{effect}' to happen, given the following input: '{input}'?",
    "Is '{effect}' a consequence of '{cause}', given the input '{input}'?",
    "Does the input '{input}' indicate '{cause}' caused '{effect}' to happen?",
    "Does the input support the claim '{cause}' caused '{effect}', input: '{input}'?",
    "Consider this input: '{input}', is '{cause}' the reason '{effect}' happened?",
    "Based on the input '{input}', is it reasonable to conclude '{cause}' lead to '{effect}' happening?",
    "If '{input}' is true, does it follow that '{cause}' resulted in '{effect}'?",
    "After examining the input '{input}', should we infer '{cause}' lead to '{effect}'?",
    "From the following input: '{input}', can we reasonably argue '{cause}' triggered the occurrence of '{effect}'?",
    "Considering this input: '{input}', does '{effect}' explicitly arise due to '{cause}'?",
    "Given this input: '{input}', is it accurate to say '{cause}' produced '{effect}'?",
    "If we accept the following information: '{input}', does this imply that '{effect}' happened specifically because '{cause}' occurred?",
    "Can we infer from the following input: '{input}' that '{cause}' was the primary factor leading to '{effect}'?",
    "Given the scenario '{input}', did '{effect}' occur as a direct consequence of '{cause}'?",
    "If we rely on this input: '{input}', is '{cause}' what specifically caused '{effect}' to occur?",
    "According to this: '{input}', was '{cause}' instrumental in bringing about '{effect}'?",
    "Analyzing this information: '{input}', can we determine whether '{effect}' was initiated by '{cause}'?",
    "Is '{cause}' explicitly identified as the trigger for '{effect}', according to the input: '{input}'?"
]

# iterate over each of the unique examples, and paraphrase with every format in P_train
new_df = {'input': [], 'label': [], 'format_idx': [], 'example_idx': []}
for _, row in T_paraphrase_unique_examples.iterrows():
    for idx, fmt in enumerate(P_train):
        new_df['input'].append(fmt.format(input=row['sentence'], cause=row['cause'], effect=row['effect']))
        new_df['label'].append(row['label'])
        new_df['format_idx'].append(idx)
        new_df['example_idx'].append(row['Unnamed: 0'])

# output T_paraphrase.csv
train_data_df = pd.DataFrame(new_df)
train_data_df.to_csv('T_paraphrase.csv')

In [ ]:
eval_data = pd.read_csv('V_standard.csv') # load V_standard

P_eval = [
    "Taking the statement '{input}' into account, is '{cause}' the event that made '{effect}' occur?",
    "If '{input}' holds true, can we definitively state that '{effect}' was brought about by '{cause}'?",
    "Does '{cause}' directly explain why '{effect}' took place, based on the information: '{input}'?",
    "Judging from '{input}', is it correct to link the occurrence of '{effect}' back to '{cause}'?",
    "If we consider '{input}', does it logically suggest that '{cause}' is behind the event '{effect}'?",
    "Based on your understanding of '{input}', would you say that '{cause}' instigated '{effect}'?",
    "Reviewing '{input}', is it valid to claim '{cause}' was the catalyst for '{effect}'?",
    "Does the provided scenario '{input}' imply that the event '{effect}' was the outcome caused by '{cause}'?"
]

# iterate over the examples in eval_data, and randomly select a paraphrase format from P_eval
new_inputs = []
format_used = []
for idx, row in eval_data.iterrows():
    choice = random.randint(0,7)
    word = P_eval[choice].format(input=row['sentence'], cause=row['cause'], effect=row['effect'])
    new_inputs.append(word)
    format_used.append(choice)
eval_data['input'] = new_inputs
eval_data['format_index'] = format_used

eval_data.to_csv('V_paraphrase.csv') # output V_paraphrase